In [2]:
# 필요한 라이브러리 설치 (처음 실행 시)
# !pip install openai-whisper tqdm

import os
import sys
import whisper
from tqdm import tqdm
import subprocess
import datetime

# 1. 기본 설정
# ==============================================================================
video_dir = r"C:\Temp\ti_movie"
output_dir = os.path.join(video_dir, "transcripts")

# ▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼
# --- ✨ 여기서 원하는 작업 모드를 선택하세요! ✨ ---
# "txt"  : 전체 대본 텍스트 파일(.txt)만 생성합니다.
# "srt"  : 싱크가 맞는 자막 파일(.srt)만 생성합니다.
# "both" : .txt 파일과 .srt 파일을 둘 다 생성합니다.
OUTPUT_MODE = "srt"
# ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲
# ==============================================================================


# --- 디렉토리 및 모델 준비 (이하 코드는 수정할 필요 없습니다) ---

# 디렉토리 존재 여부 확인 및 생성
if not os.path.exists(video_dir):
    print(f"❌ 비디오 디렉토리가 존재하지 않습니다: {video_dir}")
    sys.exit(1)
os.makedirs(output_dir, exist_ok=True)

# OUTPUT_MODE 유효성 검사
if OUTPUT_MODE not in ["txt", "srt", "both"]:
    print(f"❌ 잘못된 OUTPUT_MODE 설정입니다: '{OUTPUT_MODE}'")
    print("  'txt', 'srt', 'both' 중에서 하나를 선택해주세요.")
    sys.exit(1)

# Whisper 모델 불러오기
try:
    print("🤖 Whisper 모델을 로딩중...")
    model = whisper.load_model("small")
    print("✅ 모델 로딩 완료!")
except Exception as e:
    print(f"❌ Whisper 모델 로딩 실패: {e}")
    sys.exit(1)

# 폴더 내 영상 파일 탐색
try:
    video_files = [f for f in os.listdir(video_dir)
                   if f.lower().endswith(('.mp4', '.mkv', '.avi', '.mov', '.wmv', '.flv', '.ts'))]
    if not video_files:
        print(f"❌ {video_dir} 폴더에서 영상 파일을 찾을 수 없습니다.")
        sys.exit(1)
    print(f"📁 발견된 영상 파일: {len(video_files)}개")
    for i, file in enumerate(video_files, 1):
        print(f"  {i}. {file}")
    print()
except Exception as e:
    print(f"❌ 파일 탐색 중 오류 발생: {e}")
    sys.exit(1)

# --- 영상 처리 시작 ---
for video_file in tqdm(video_files, desc="🎧 Processing videos"):
    try:
        video_path = os.path.join(video_dir, video_file)
        base_name = os.path.splitext(video_file)[0]
        audio_path = os.path.join(video_dir, f"{base_name}.wav")
        
        # 건너뛰기 로직: 각 모드에 맞게 이미 결과물이 있는지 확인
        should_skip = False
        if OUTPUT_MODE == "txt":
            if os.path.exists(os.path.join(output_dir, f"{base_name}.txt")):
                should_skip = True
        elif OUTPUT_MODE == "srt":
            if os.path.exists(os.path.join(output_dir, f"{base_name}.srt")):
                should_skip = True
        elif OUTPUT_MODE == "both":
            if os.path.exists(os.path.join(output_dir, f"{base_name}.txt")) and \
               os.path.exists(os.path.join(output_dir, f"{base_name}.srt")):
                should_skip = True
        
        if should_skip:
            print(f"\n⏭️ 이미 처리된 파일입니다: {video_file}")
            continue

        print(f"\n🎬 처리중: {video_file}")

        # 1. 영상에서 오디오 추출 (FFmpeg)
        print("  🔊 오디오 추출중...")
        command = [
            'ffmpeg', '-i', video_path, '-vn', '-acodec', 'pcm_s16le',
            '-ar', '16000', '-ac', '1', '-y', audio_path
        ]
        subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
        # 2. Whisper로 음성 인식
        print("  🗣️ 음성 인식중...")
        result = model.transcribe(audio_path, language="en")

        # 3. 선택된 모드에 따라 파일 생성
        # TXT 모드 또는 BOTH 모드일 경우
        if OUTPUT_MODE in ["txt", "both"]:
            txt_path = os.path.join(output_dir, f"{base_name}.txt")
            with open(txt_path, "w", encoding="utf-8") as f:
                f.write(result["text"])
            print(f"  ✅ 텍스트(.txt) 저장 완료: {txt_path}")

        # SRT 모드 또는 BOTH 모드일 경우
        if OUTPUT_MODE in ["srt", "both"]:
            srt_path = os.path.join(output_dir, f"{base_name}.srt")
            with open(srt_path, "w", encoding="utf-8") as srt_file:
                for i, segment in enumerate(result['segments']):
                    start_time = segment['start']
                    end_time = segment['end']
                    text = segment['text']
                    start_srt = str(datetime.timedelta(seconds=int(start_time))) + f",{int((start_time % 1) * 1000):03d}"
                    end_srt = str(datetime.timedelta(seconds=int(end_time))) + f",{int((end_time % 1) * 1000):03d}"
                    srt_file.write(f"{i + 1}\n")
                    srt_file.write(f"{start_srt} --> {end_srt}\n")
                    srt_file.write(f"{text.strip()}\n\n")
            print(f"  ✅ 자막(.srt) 저장 완료: {srt_path}")

        # 4. 임시 오디오 파일 삭제
        if os.path.exists(audio_path):
            os.remove(audio_path)

    except subprocess.CalledProcessError:
        print(f"  ❌ 오류 ({video_file}): 오디오 트랙을 추출할 수 없습니다.")
        continue
    except Exception as e:
        print(f"  ❌ 오류 ({video_file}): {str(e)}")
        if 'audio_path' in locals() and os.path.exists(audio_path):
            try: os.remove(audio_path)
            except: pass
        continue

print("\n🎉 모든 작업이 완료되었습니다!")
print(f"📂 결과 파일 위치: {output_dir}")

🤖 Whisper 모델을 로딩중...
✅ 모델 로딩 완료!
📁 발견된 영상 파일: 11개
  1. 2025 BCA Update and Titanium Perspective TITANIUM USA 2025 Bosto.mp4
  2. Airbus Global Market Forecast.ts
  3. Defense-Moderaor_Viv Helwig Vested Metals International.mp4
  4. Fundamentals of Titanium Workshop.mp4
  5. Global Industrial Markets - Moderator_Nate Fairfield, Titanium Industries.ts
  6. Manufacturing Techologies_ Moderator Tirthesh Ingale, University of North Texas.mp4
  7. Safety Education Committee update on NFPA 660 TITANIUM USA 2025.mp4
  8. Titanium Industry World Supply Trends Moderator Andy Bayne, TIMET.ts
  9. Titanium Technology in Medical Applications Moderator Colin McCracken, Oerlikon Metco Canada Inc.ts
  10. World Demand Trends-Ryosaku Kadowaki Presentation.ts
  11. World Titanium Industry Demand Trends Moderator Peter Zimm Charl.mp4



🎧 Processing videos:   0%|          | 0/11 [00:00<?, ?it/s]


🎬 처리중: 2025 BCA Update and Titanium Perspective TITANIUM USA 2025 Bosto.mp4
  🔊 오디오 추출중...
  🗣️ 음성 인식중...


🎧 Processing videos:   9%|▉         | 1/11 [09:59<1:39:55, 599.59s/it]

  ✅ 자막(.srt) 저장 완료: C:\Temp\ti_movie\transcripts\2025 BCA Update and Titanium Perspective TITANIUM USA 2025 Bosto.srt

🎬 처리중: Airbus Global Market Forecast.ts
  🔊 오디오 추출중...
  🗣️ 음성 인식중...


...

  ✅ 자막(.srt) 저장 완료: C:\Temp\ti_movie\transcripts\Titanium Technology in Medical Applications Moderator Colin McCracken, Oerlikon Metco Canada Inc.srt

🎬 처리중: World Demand Trends-Ryosaku Kadowaki Presentation.ts
  🔊 오디오 추출중...
  🗣️ 음성 인식중...


🎧 Processing videos:  91%|█████████ | 10/11 [2:07:00<11:04, 664.43s/it]

  ✅ 자막(.srt) 저장 완료: C:\Temp\ti_movie\transcripts\World Demand Trends-Ryosaku Kadowaki Presentation.srt

🎬 처리중: World Titanium Industry Demand Trends Moderator Peter Zimm Charl.mp4
  🔊 오디오 추출중...
  🗣️ 음성 인식중...


🎧 Processing videos: 100%|██████████| 11/11 [2:41:51<00:00, 882.83s/it] 

  ✅ 자막(.srt) 저장 완료: C:\Temp\ti_movie\transcripts\World Titanium Industry Demand Trends Moderator Peter Zimm Charl.srt

🎉 모든 작업이 완료되었습니다!
📂 결과 파일 위치: C:\Temp\ti_movie\transcripts
